In [ ]:
from src import data_loader, features, volatility, regimes, models_xgb

# 1. Ingest & Transform
raw_df = data_loader.fetch_raw_data()
stat_df = features.engineer_stationary_features(raw_df)
garch_df = volatility.generate_expanding_garch(stat_df)

# 2. HMM Target Generation (Now immune to Label Switching)
hmm_df = regimes.generate_expanding_hmm_targets(
    garch_df,
    min_window=504,
    feature_cols=['Log_Return', 'Vol_EGARCH'], 
    vol_col='Vol_EGARCH',
    n_components=3
)

# 3. Merge & Lag
master_df = garch_df.join(hmm_df, how='inner')
cols_to_lag = [
    'Log_Return', 'Sq_Log_Return', 'Vol_GARCH', 'Vol_EGARCH', 
    'VIX_Change', 'Oil_Change', 'CPI_MoM', 'FedFunds_Diff', 'Term_Spread_Diff'
]
final_df = features.create_lags(master_df, feature_cols=cols_to_lag)

# 4. Train & Evaluate
final_model, feature_names = models_xgb.train_xgb_walk_forward(
    df=final_df, 
    target_col='Target_HMM', 
    n_splits=4,
    use_mlflow=False # Set to True once you `pip install mlflow`
)

In [ ]:
# 5. Visualize
models_xgb.plot_feature_importance(final_model, feature_names)